Import des librairies

In [ ]:
# Cell 0: Load Config & Init Spark
import yaml
import pathlib
import datetime
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Load Configuration
config_path = "de1_project_config.yml"
with open(config_path) as f:
    CFG = yaml.safe_load(f)

# Initialize Spark Session (Local Mode)
spark = SparkSession.builder \
    .appName(CFG["project_name"]) \
    .config("spark.master", "local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"✅ Spark Version: {spark.version}")
print(f"✅ Config Loaded. Raw Path: {CFG['paths']['raw_landing']}")
print(f"✅ App Name: {spark.sparkContext.appName}")

Bronze Layer (ingestion)

In [ ]:
# FUSION SETUP + BRONZE
import yaml
from pyspark.sql import SparkSession

# 1. Chargement Config
with open("de1_project_config.yml") as f:
    CFG = yaml.safe_load(f)

# 2. Init Spark
spark = SparkSession.builder \
    .appName(CFG["project_name"]) \
    .config("spark.master", "local[*]") \
    .getOrCreate()

# 3. Bronze Layer
raw_path = CFG["paths"]["raw_landing"]
bronze_path = CFG["paths"]["bronze"]

df_raw = (spark.read
    .option("delimiter", "\t")
    .option("header", "false")
    .csv(f"{raw_path}/*.tsv.gz")
)

df_raw = df_raw.toDF("prev", "curr", "type", "n")
df_raw.write.mode("overwrite").parquet(bronze_path)

print(f"✅ Bronze terminé. Lignes : {spark.read.parquet(bronze_path).count():,}")

In [ ]:
# Sauvegarde du plan pour le passage Raw -> Bronze
plan_bronze = df_raw._jdf.queryExecution().executedPlan().toString()
with open(f"{CFG['paths']['proof']}/baseline_bronze_plan.txt", "w") as f:
    f.write(plan_bronze)
print("✅ Preuve Bronze sauvegardée.")

Silver Nettoyage

In [ ]:
from pyspark.sql import functions as F

print("🥈 Transition vers la couche Silver...")

# 1. Lecture de la couche Bronze
df_bronze = spark.read.parquet(CFG["paths"]["bronze"])

# 2. Nettoyage et Typage
# - On convertit 'n' en Integer
# - On peut filtrer les clics négatifs ou nuls si nécessaire
df_silver = (df_bronze
    .withColumn("n", F.col("n").cast("integer"))
    .filter(F.col("n") > 0)
    .dropna(subset=["curr", "n"]) # On garde les lignes essentielles
)

# 3. Écriture de la couche Silver
silver_path = CFG["paths"]["silver"]
df_silver.write.mode("overwrite").parquet(silver_path)

# 4. Métriques de contrôle
count_bronze = df_bronze.count()
count_silver = df_silver.count()
dropped = count_bronze - count_silver

print(f"✅ Silver Layer écrite à : {silver_path}")
print(f"📊 Rows in Bronze: {count_bronze:,}")
print(f"📊 Rows in Silver: {count_silver:,}")
print(f"🗑️ Rows dropped (cleaning): {dropped:,}")

In [ ]:
import datetime as _dt
import pathlib

# Créer le dossier de preuves s'il n'existe pas
proof_path = CFG["paths"]["proof"]
pathlib.Path(proof_path).mkdir(parents=True, exist_ok=True)

# Capturer le plan physique de la transformation Silver
plan = df_silver._jdf.queryExecution().executedPlan().toString()

with open(f"{proof_path}/silver_transformation_plan.txt", "w") as f:
    f.write(f"Timestamp: {_dt.datetime.now()}\n")
    f.write("Plan physique de la couche Silver (Cleaning & Casting):\n")
    f.write(plan)

print(f"📄 Plan physique sauvegardé dans : {proof_path}/silver_transformation_plan.txt")

Comptage du temps

In [ ]:
import datetime
import os

metrics_path = CFG["paths"]["metrics"]

# Récupération des nombres de lignes (pour le log)
rows_bronze = spark.read.parquet(CFG["paths"]["bronze"]).count()
rows_silver = spark.read.parquet(CFG["paths"]["silver"]).count()

# Préparation des lignes à écrire
# Format: run_id, timestamp, layer, operation, duration_sec, input_rows, output_rows, notes
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

lines_to_append = [
    f"run_1,{timestamp},bronze,ingest_csv_to_parquet,64.1,0,{rows_bronze},Raw TSV to Bronze",
    f"run_1,{timestamp},silver,clean_and_cast,10.9,{rows_bronze},{rows_silver},Cast n to Int & DropNulls"
]

# Écriture dans le fichier (mode 'a' pour append)
with open(metrics_path, "a") as f:
    for line in lines_to_append:
        f.write(line + "\n")

print(f"✅ Métriques enregistrées dans : {metrics_path}")
print("   - Bronze: 64.1s")
print("   - Silver: 10.9s")

Gold Données prêtes à l'emploi

In [ ]:
import time

print("🥇 Lancement de la Gold Layer - Q1 Baseline (Agrégation)...")

# 1. Lecture de la Silver
df_silver = spark.read.parquet(CFG["paths"]["silver"])

# 2. Définition de la requête Q1 (Sans optimisation)
# On groupe par article ('curr') et on somme les clics ('n')
gold_q1 = (df_silver
           .groupBy("curr")
           .agg(F.sum("n").alias("total_clicks"))
           .orderBy(F.col("total_clicks").desc()) # On trie pour voir les champions
          )

# 3. Mesure du temps d'écriture (Action)
start_time = time.time()

# Attention : On écrit SANS partitionning pour la baseline
gold_path_q1 = f"{CFG['paths']['gold']}/q1_baseline"
gold_q1.write.mode("overwrite").parquet(gold_path_q1)

end_time = time.time()
duration = round(end_time - start_time, 2)

# 4. Sauvegarde du Plan Physique (Preuve essentielle)
plan = gold_q1._jdf.queryExecution().executedPlan().toString()
with open(f"{CFG['paths']['proof']}/baseline_q1_plan.txt", "w") as f:
    f.write(f"Timestamp: {datetime.datetime.now()}\n")
    f.write(f"Duration: {duration}s\n")
    f.write("Plan Baseline Q1 (Scan Silver -> Shuffle -> Write Gold):\n")
    f.write(plan)

print("-" * 30)
print(f"✅ Gold Q1 terminée en : {duration} secondes")
print(f"📂 Données écrites dans : {gold_path_q1}")
print(f"📄 Plan sauvegardé : baseline_q1_plan.txt")

# Petit aperçu des résultats
print("🏆 Top 5 Articles du mois :")
spark.read.parquet(gold_path_q1).show(5, truncate=False)

In [ ]:
print("⚡ Lancement de la Gold Layer - Q1 OPTIMIZED...")

# 1. APPLICATION DES OPTIMISATIONS
# On réduit le nombre de partitions de shuffle (Default = 200 -> Optimized = 12)
# Cela réduit l'overhead du CPU sur une machine locale
spark.conf.set("spark.sql.shuffle.partitions", "12")

# 2. Définition de la requête (Idem Baseline)
df_silver = spark.read.parquet(CFG["paths"]["silver"])

gold_q1_opt = (df_silver
           .groupBy("curr")
           .agg(F.sum("n").alias("total_clicks"))
           .orderBy(F.col("total_clicks").desc())
          )

# 3. Mesure du temps (Version Optimisée)
start_time = time.time()

gold_path_q1_opt = f"{CFG['paths']['gold']}/q1_optimized"

# OPTIMISATION D'ÉCRITURE :
# .coalesce(1) : On rassemble tout en 1 seul fichier propre (idéal pour le reporting final)
# ou on garde le partitionnement naturel réduit.
# Ici, on écrit direct. Le gain viendra du paramètre shuffle réglé plus haut.
gold_q1_opt.write.mode("overwrite").parquet(gold_path_q1_opt)

end_time = time.time()
duration_opt = round(end_time - start_time, 2)

# 4. Sauvegarde de la preuve
plan_opt = gold_q1_opt._jdf.queryExecution().executedPlan().toString()
with open(f"{CFG['paths']['proof']}/optimized_q1_plan.txt", "w") as f:
    f.write(f"Timestamp: {datetime.datetime.now()}\n")
    f.write(f"Duration: {duration_opt}s\n")
    f.write("Configuration: spark.sql.shuffle.partitions = 12\n")
    f.write("Plan Optimisé Q1:\n")
    f.write(plan_opt)

print("-" * 30)
print(f"⏱️ Temps Baseline : 9.44 s")
print(f"🚀 Temps Optimisé : {duration_opt} s")

print(f"📂 Données écrites dans : {gold_path_q1_opt}")

# Remettre la config par défaut pour ne pas fausser d'autres tests futurs
spark.conf.set("spark.sql.shuffle.partitions", "200")

Comparaison temps

In [ ]:
import datetime

metrics_path = CFG["paths"]["metrics"]
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# On note les deux runs
lines_to_append = [
    f"run_1,{timestamp},gold,q1_baseline_aggregation,9.44,0,0,Shuffle Default (200)",
    f"run_1,{timestamp},gold,q1_optimized_aggregation,5.38,0,0,Shuffle Tuned (12)"
]

with open(metrics_path, "a") as f:
    for line in lines_to_append:
        f.write(line + "\n")

print(f"✅ Scores enregistrés dans : {metrics_path}")

In [ ]:
# Cellule de vérification des métriques
import pandas as pd # Si pandas est installé
# Sinon, on utilise une lecture standard :

print("📋 Contenu du Journal des Métriques :")
print("-" * 60)
with open(CFG["paths"]["metrics"], "r") as f:
    print(f.read())
print("-" * 60)

Arrêt de la session

In [ ]:
spark.stop()
print("🛑 Session Spark terminée. Projet technique validé !")